# DAS Çit-İhlali Sınıflandırması — Colab Eğitimi

2D-CNN + SK-Attention. Bu notebook **yalnızca eğitim ve değerlendirme** yapar (Faz 3–4).

## Önemli: bölmeler burada YENİDEN ÜRETİLMEZ

Faz 0 ve Faz 1 yerelde çalıştırıldı; `outputs/metadata.csv` ve `outputs/folds/fold_*.json`
repoda hazır duruyor. Bu notebook onları **okur**.

Sebep: bölme mantığının tamamı (kaynak grupları, MixUp eleme kuralı, sızıntı testleri)
tek bir yerde ve versiyon kontrollü. Colab kendi bölmesini üretseydi, farklı bir
scikit-learn sürümü farklı katmanlar doğurabilir ve sonuçlar yereldekiyle
karşılaştırılamaz hale gelirdi.

## Çalıştırma sırası

Menüden **Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU** seçtiğinden emin ol,
sonra hücreleri sırayla çalıştır.

## 1. Repoyu klonla

In [ ]:
import os

REPO = 'https://github.com/salihcengiz/wav-dataset.git'
if not os.path.isdir('/content/wav-dataset'):
    !git clone --depth 1 {REPO} /content/wav-dataset
%cd /content/wav-dataset

# Bu hucreyi TEKRAR calistirirsan repo en son surume guncellenir.
# (Klasor zaten varken 'git clone' hata verirdi -- bu yuzden fetch+reset.)
!git fetch --depth 1 -q origin main && git reset --hard -q origin/main
!git log --oneline -1

## 2. Ortamı hazırla

Colab'da `torch`, `torchvision`, `scikit-learn`, `pandas`, `matplotlib`, `seaborn`
zaten kurulu gelir. Aşağıdaki hücre sadece **sürümleri ve GPU'yu doğrular**.

In [ ]:
import sys, torch, sklearn, pandas, numpy
sys.path.insert(0, 'src')

print('python      ', sys.version.split()[0])
print('torch       ', torch.__version__)
print('torchvision ', __import__('torchvision').__version__)
print('sklearn     ', sklearn.__version__)
print('pandas      ', pandas.__version__)
print()
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0),
          f"({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)")
else:
    print('!!! GPU YOK -- Calisma zamani > Calisma zamani turunu degistir > T4 GPU')

## 3. Veri ve bölmeleri doğrula

Repodan gelen `metadata.csv` ve fold JSON'larının tutarlı olduğunu kontrol eder.
Faz 1'in sızıntı garantileri burada tekrar sınanır — Colab'a yanlış bir bölme
geldiyse eğitime başlamadan önce durmak gerekir.

In [ ]:
import json, pandas as pd
import config as cfg

df = pd.read_csv(cfg.METADATA_CSV)
print(f'metadata.csv : {len(df)} satir')
print(f'  tek-kaynakli {(~df.is_mixup).sum()} | mixup {df.is_mixup.sum()}')

groups = set(df.group_1) | set(df.group_2.dropna())
print(f'  etkin bagimsiz kayit: {len(groups)}  (beklenen {cfg.N_EFFECTIVE_GROUPS})')
assert len(groups) == cfg.N_EFFECTIVE_GROUPS

print()
for i in range(cfg.N_SPLITS):
    f = json.loads((cfg.FOLDS_DIR / f'fold_{i}.json').read_text(encoding='utf-8'))
    tr, va, te = f['train_idx'], f['val_idx'], f['test_idx']
    # Sizinti testleri (Faz 1'in garantileri Colab'da da gecerli mi)
    assert not (set(tr) & set(va)) and not (set(tr) & set(te)) and not (set(va) & set(te))
    assert not df.is_mixup.to_numpy()[te].any(), f'fold {i}: testte mixup var'
    assert not df.is_mixup.to_numpy()[va].any(), f'fold {i}: dogrulamada mixup var'
    assert not (set(f['train_groups']) & set(f['test_groups']))
    print(f"  fold {i}: egitim {f['n_train']:>4} | dogrulama {f['n_val']:>4} | "
          f"test {f['n_test']:>4} | atilan mixup {f['n_dropped_mixup']:>4}")
print('\n[x] Tum sizinti testleri gecti')

## 4. Model birim testi

Faz 2'nin kabul kriterlerini GPU ortamında tekrar koşturur.

In [ ]:
!python src/model.py

## 5. Veri yükleyici testi + artırma önizlemesi

959 PNG'yi RAM'e alır (~460 MB) ve artırmanın spektrograma ne yaptığını gösterir.
Üretilen görseli **gözle incele**: dikey (zaman) ve yatay (frekans) maskeleme
şeritleri görünmeli, renkler bozulmamalı, flip olmamalı.

In [ ]:
!python src/dataset.py

In [ ]:
from IPython.display import Image, display
display(Image(str(cfg.FIG_DIR / 'augmentation_preview.png'), width=760))

## 6. Tek katman — kayıp eğrisini incele

**Plan 12.4 bunu özellikle söylüyor:** tüm katmanları çalıştırmadan önce tek bir
katmanı eğit ve aşırı öğrenme var mı bak.

Bakılacak şey: eğitim ve doğrulama eğrileri ciddi şekilde ayrışıyor mu (Plan 7.6).

In [ ]:
!python src/train.py --fold 0 --attention sk

In [ ]:
display(Image(str(cfg.FIG_DIR / 'fold_0_sk_curves.png'), width=1100))

## 7. Tüm katmanlar — ana model (SK-Attention)

Yukarıdaki eğri makul görünüyorsa devam et.

In [ ]:
!python src/train.py --fold all --attention sk

## 8. Ablasyon — SK'siz baseline (ve isteğe bağlı SE / CBAM)

**Aynı bölmelerle** eğitilir; karşılaştırma ancak böyle anlamlıdır.

In [ ]:
!python src/train.py --fold all --attention none

In [ ]:
# Istege bagli (PLAN 6.3 bunlari 'istege bagli' isaretliyor)
!python src/train.py --fold all --attention se
!python src/train.py --fold all --attention cbam

## 9. Sonuçları Drive'a yedekle

⚠️ Colab oturumu koptuğunda `/content` altındaki her şey silinir.
Checkpoint'leri, eğrileri ve sonuç JSON'larını Drive'a kopyala.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, datetime
stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M')
dest = f'/content/drive/MyDrive/das_outputs/{stamp}'
shutil.copytree('outputs', dest, dirs_exist_ok=True)
print('kaydedildi ->', dest)
!du -sh {dest}

## 10. Faz 4 — Değerlendirme

`src/evaluate.py` henüz yazılmadı (Faz 4). Yazıldığında bu hücre çalıştırılacak:
macro-F1 (ort ± std), sınıf bazında precision/recall/F1, karışıklık matrisi,
SNR kırılımı ve t-SNE (baseline vs SK).

In [ ]:
# !python src/evaluate.py